# Drift mùa vụ và drift COVID trên NYC TLC

Bốn tháng `2019-07`, `2019-12`, `2020-02`, `2020-04`, mỗi tháng mẫu 100 000 chuyến (`seed=0`). Điền 6 hàm trong các cell chấm điểm; cell `explore` chỉ để xem.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

MONTHS = {
    "2019-07": "nyc-tlc",
    "2019-12": "nyc-tlc",
    "2020-02": "nyc-tlc-2020",
    "2020-04": "nyc-tlc-2020",
}
SAMPLE_N = 100_000
SEED = 0
COLUMNS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount", "passenger_count", "PULocationID"]
FEATURES = ["trip_distance", "fare_amount", "hour"]
MODEL_FEATURES = ["trip_distance", "hour", "PULocationID"]

In [ ]:
def load_sample(path, n=SAMPLE_N, seed=SEED):
    # TODO: chỉ đọc COLUMNS, lấy mẫu n dòng xác định theo seed (DataFrame.sample(random_state=seed))
    raise NotImplementedError

In [ ]:
def prepare(df):
    # TODO: thêm duration_min (phút) và hour, giữ dòng theo 4 luật trong README, reset_index(drop=True)
    raise NotImplementedError

In [ ]:
def psi(reference, current, bins=10, eps=1e-4):
    # TODO: PSI với biên bin là quantile của reference, bỏ NaN, chặn dưới tỉ lệ bằng eps (luật trong README)
    raise NotImplementedError

In [ ]:
def psi_by_column(reference, current, columns=FEATURES):
    # TODO: dict {cột: PSI(reference[cột], current[cột])}
    raise NotImplementedError

In [ ]:
def missing_share(df, column):
    # TODO: tỉ lệ NaN của một cột
    raise NotImplementedError

In [ ]:
def eta_mae(reference, current, seed=SEED):
    # TODO: train HistGradientBoostingRegressor trên 80% reference (luật trong README), trả (MAE holdout 20% reference, MAE trên current)
    raise NotImplementedError

In [ ]:
# explore: tải dữ liệu nếu chưa có và đọc bốn tháng
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

data = {}
for month, dataset in MONTHS.items():
    path = ensure(dataset) / f"yellow_tripdata_{month}.parquet"
    data[month] = prepare(load_sample(path))
    print(month, len(data[month]), "dòng sau lọc")

In [ ]:
# explore: PSI từng feature cho hai cặp tháng
pairs = {"mùa vụ 2019-07 -> 2019-12": ("2019-07", "2019-12"), "COVID 2020-02 -> 2020-04": ("2020-02", "2020-04")}
table = pd.DataFrame({name: psi_by_column(data[a], data[b]) for name, (a, b) in pairs.items()})
print(table.round(4))
for name, (a, b) in pairs.items():
    print(name, "passenger_count NaN:", round(missing_share(data[a], "passenger_count"), 4), "->", round(missing_share(data[b], "passenger_count"), 4))

In [ ]:
# explore: chất lượng model ETA huấn luyện trên tháng đầu của mỗi cặp
for name, (a, b) in pairs.items():
    holdout, current = eta_mae(data[a], data[b])
    print(name, "MAE holdout", round(holdout, 3), "MAE tháng sau", round(current, 3), "tăng", f"{current / holdout - 1:.1%}")

In [ ]:
# explore: biểu đồ PSI hai cặp tháng
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

ax = table.plot.bar(rot=0)
ax.axhline(0.1, color="gray", linestyle="--")
ax.set_ylabel("PSI")
plt.tight_layout()
plt.savefig("psi_seasonal_vs_covid.png")

Làm phần trắc nghiệm trong app sau khi chạy notebook.